In [1]:
import numpy as np
from metavision_core.event_io.raw_reader import RawReader
from matplotlib import pyplot as plt
from scipy import sparse
from tqdm import tqdm
import os
import gc
import cv2
from skimage.transform import warp
from skimage.registration import optical_flow_tvl1, optical_flow_ilk
from skimage.registration import phase_cross_correlation
from parallel_merge import process_line_by_line_n_deg

In [2]:
"""
BLOCK: Read events from trigger mode
"""
file_name = ""
axis = "x" # x or y
layer = 0 # 0, 1, 2
scale_factor = 2
interval = 180
raw_stream_x = RawReader("../data/focal_stack/"+file_name+axis+".raw", max_events=int(8e9))
output_folder = "../results/focal_stack/"+str(layer)+"/"
events = raw_stream_x.load_n_events(int(8e9-1e6))
fs_start = round(360+interval*(layer-1))
print(fs_start)
num_lines = 16
# BLOCK: IF Focal stack
y_min = int(fs_start)
y_max = int(fs_start + num_lines)
events = events[(events['y'] >= y_min) & (events['y'] <= y_max)]
external_triggers_x = raw_stream_x.get_ext_trigger_events()
positive_triggers_x = external_triggers_x[external_triggers_x['p']==1]

print("Number of events loaded:", len(events))
print(f"Total number of external triggers: {len(external_triggers_x)}")
print(f"Number of positive external triggers: {len(positive_triggers_x)}")
print("First 4 external triggers:", external_triggers_x[:4])
print("First 2 positive external triggers:", positive_triggers_x[:2])

trigger_times = np.array([trigger['t'] for trigger in positive_triggers_x[:25]])
trigger_differences = np.diff(trigger_times)
print("Differences between consecutive triggers:", trigger_differences)
print("Time for one line scan:", trigger_times[-1]-trigger_times[0])

# pixel_per_mm = int(1225/0.18)
pixel_per_mm = 5760
mm_per_move = 0.08 #mm
deg = np.degrees(np.arctan((21 + mm_per_move * pixel_per_mm)/(mm_per_move * pixel_per_mm)))
# deg = 45
print(deg)
if axis == "y":
    deg = 90 - deg
cos_deg = np.cos(deg * np.pi / 180)
sin_deg = np.sin(deg * np.pi / 180)
total_scan_area = 6 + 0.8 #mm
num_trigger_per_mm = 10
num_trigger_per_line = round((total_scan_area - 6) * num_trigger_per_mm + 1) #number of external triggers per line
fully_reconstruction_area = total_scan_area-6 #mm
# merged_height = pixel_per_mm*total_scan_area + 360*2
merged_width = round(pixel_per_mm*fully_reconstruction_area + 640*2)
trigger_interval = fully_reconstruction_area/(num_trigger_per_line-1) #mm
scanning_sensor_width = 1000
print(fully_reconstruction_area / mm_per_move)
assert len(positive_triggers_x) == int(num_trigger_per_line * round(fully_reconstruction_area / mm_per_move))

merged_height = int(np.ceil((scanning_sensor_width + np.max(events['y'])-np.min(events['y'])) * cos_deg))
print(np.max(events['y'])-np.min(events['y']))
# merged_height = int(np.ceil((scanning_sensor_width + 40) * cos_deg))
print(merged_height)
overlap_width = int(merged_height - mm_per_move * pixel_per_mm) // scale_factor
print(overlap_width)
# assert overlap_width > 50

if axis == "x":
    events_y_coord = events['x'] - np.min(events['x'])
else:
    events_y_coord = np.max(events['x']) - events['x']

lines_x = process_line_by_line_n_deg(
    positive_triggers_t=positive_triggers_x['t'],
    events_t=events['t'],
    
    events_x_coord=events['y'] - np.min(events['y']), 
    events_y_coord=events_y_coord, 

    events_p=events['p'],
    total_lines=round(fully_reconstruction_area / mm_per_move),
    num_trigger_per_line=num_trigger_per_line,
    mm_per_move=mm_per_move,
    pixel_per_mm=pixel_per_mm,
    trigger_interval_x=trigger_interval,
    merged_height=merged_width,
    merged_width=merged_height,
    manual_shift=0,
    axis=axis,
    scale_factor=scale_factor, 
    deg=deg) 

180
Number of events loaded: 370534622
Total number of external triggers: 180
Number of positive external triggers: 90
First 4 external triggers: [(0, 3051635, 0) (1, 3051889, 0) (0, 3061388, 0) (1, 3061637, 0)]
First 2 positive external triggers: [(1, 3051889, 0) (1, 3061637, 0)]
Differences between consecutive triggers: [   9748    9998    9751   10248   10247    9753    9999    9999 1763948
    9752   10010    9993   10258    9984   10001   10002    9996 1864697
   10000    9748    9995   10252   10002    9995]
Time for one line scan: 3848376
46.27627027338357
9.999999999999998
16
703
121


In [3]:
# average corse shift calculation
average_shift_odd = 0
average_shift_even = 0
average_overlap_width = 0
count_invalid_odd = 0
count_invalid_even = 0
count_invalid_y = 0
for i in tqdm(range(0, len(lines_x)-1, 1)):
    line_x1 = lines_x[i][:, -overlap_width-1:-1].copy()
    line_x2 = lines_x[i+1][:, 0:overlap_width].copy()
    line_x1 = np.where(np.abs(line_x1) > 2, line_x1, 0)
    line_x2 = np.where(np.abs(line_x2) > 2, line_x2, 0)
    shift, error, diffphase = phase_cross_correlation(line_x1, line_x2, reference_mask=line_x1 != 0, moving_mask=line_x2 != 0)
    print(f"shift: {shift}")
    if np.abs(shift[0]) > 50:
        if i % 2 == 0:
            count_invalid_odd += 1
        else:
            count_invalid_even += 1
        continue
    if i % 2 == 0:
        average_shift_odd += shift[0]
    else:
        average_shift_even += shift[0]
    if np.abs(shift[1]) > 30:
        count_invalid_y += 1
        continue
    average_overlap_width += shift[1]
average_shift_odd /= len(lines_x) // 2 - count_invalid_odd  
average_shift_even /= len(lines_x) // 2 - count_invalid_even - ((len(lines_x)-1) % 2)
average_overlap_width /= len(lines_x) - count_invalid_odd - count_invalid_even - count_invalid_y
overlap_width = int(overlap_width - average_overlap_width)
print(f"average_shift_odd: {average_shift_odd}")
print(f"average_shift_even: {average_shift_even}")
print(f"average_overlap_width: {average_overlap_width}")
print(f"overlap_width: {overlap_width}")
print(f"count_invalid_odd: {count_invalid_odd}")
print(f"count_invalid_even: {count_invalid_even}")
print(f"count_invalid_y: {count_invalid_y}")
assert overlap_width > 30

for i in tqdm(range(0, len(lines_x)-1, 2)):
    lines_x[i+1] = np.roll(lines_x[i+1], int(average_shift_odd), axis=0)


for i in tqdm(range(0, len(lines_x), 1)):
    lines_x[i] = lines_x[i][:, 20:-20]
    # print(lines_x[i].shape)

overlap_width = int(overlap_width - 40)
print(overlap_width)
assert overlap_width > 30


 11%|█         | 1/9 [00:00<00:02,  3.41it/s]

shift: [-18.   1.]


 22%|██▏       | 2/9 [00:00<00:02,  3.45it/s]

shift: [13. -3.]


 33%|███▎      | 3/9 [00:00<00:01,  3.46it/s]

...

shift: [ 8. -5.]


100%|██████████| 9/9 [00:02<00:00,  3.46it/s]


shift: [-21.   3.]
average_shift_odd: -19.8
average_shift_even: 7.5
average_overlap_width: -1.0
overlap_width: 122
count_invalid_odd: 0
count_invalid_even: 0
count_invalid_y: 0


100%|██████████| 10/10 [00:00<00:00, 330260.16it/s]

82


In [4]:
# DEBUG: Plot all event lines
line_width_x = merged_height // scale_factor - 40
os.makedirs(output_folder, exist_ok=True)

empty_events = np.zeros((merged_width // scale_factor, line_width_x*round(fully_reconstruction_area / mm_per_move)), dtype=np.int8)
print(empty_events.shape)
print(line_width_x)
print(len(lines_x))
for i, line_x in enumerate(lines_x):
    # print(line_width_x * i, line_width_x * (i + 1))
    empty_events[:, line_width_x * i : line_width_x * (i + 1)] = line_x

# np.save(os.path.join(output_folder, f"all_events.npy"), empty_events)
plt.imsave(os.path.join(output_folder, f"all_events_{axis}.png"), empty_events[::2, ::2], vmin=-5, vmax=5)
plt.imsave(os.path.join(output_folder, f"exapmle_events_{axis}.png"), lines_x[5], vmin=-5, vmax=5)

(2944, 3110)
311
10


In [5]:
del events
del raw_stream_x
# del empty_events
gc.collect()

20264

In [7]:
merged_events_x = lines_x[0]

for i in tqdm(range(round(round(fully_reconstruction_area / mm_per_move) - 1)), desc="Merging events"):
    # line_index_1 = i
    line_index_2 = i + 1

    line_x1 = merged_events_x[:, -overlap_width-1:-1].copy()
    line_x2 = lines_x[line_index_2][:, 0:overlap_width].copy()
    line_x1 = np.where(np.abs(line_x1) > 2, line_x1, 0)
    line_x2 = np.where(np.abs(line_x2) > 2, line_x2, 0)
    # line_x1 = np.where(np.abs(line_x1) > 1, np.abs(line_x1), 0)
    # line_x2 = np.where(np.abs(line_x2) > 1, np.abs(line_x2), 0)

    # shift, error, diffphase = phase_cross_correlation(line_x1, line_x2)
    # if shift[0] == 0 and shift[1] == 0:
    #     continue
    line_x1 = line_x1 // 2
    line_x2 = line_x2 // 2
    line_x1 = np.clip(line_x1, -8, 8)
    line_x2 = np.clip(line_x2, -8, 8)

    shift, error, diffphase = phase_cross_correlation(line_x1, line_x2)
    print(f"shift: {shift}, error: {error}, diffphase: {diffphase}")

    # unique_vals, counts = np.unique(line_x1, return_counts=True)

    
    if np.abs(shift[0]) > 200:
        print(f"unreasonable shift_x: {shift}, resetting to 0")
        shift[0] = 0
    if np.abs(shift[1]) > 20:
        print(f"unreasonable shift_y: {shift}, resetting to 0")
        shift[1] = 0

  
    shifted_line_x2 = np.roll(line_x2, int(shift[0]), axis=0)
    shifted_x2 = np.roll(lines_x[line_index_2], int(shift[0]), axis=0)
    # shifted_line_x2 = line_x2
    # shifted_x2 = lines_x[line_index_2]

    line_x1 = line_x1 - 9
    shifted_line_x2 = shifted_line_x2 - 9
    line_x1 = np.where(line_x1 != -9, line_x1, 0)
    shifted_line_x2 = np.where(shifted_line_x2 != -9, shifted_line_x2, 0)
    line_x1 = line_x1.astype(np.uint8)
    shifted_line_x2 = shifted_line_x2.astype(np.uint8)

    shift, error, diffphase = phase_cross_correlation(line_x1, shifted_line_x2, upsample_factor=10, reference_mask=line_x1 != 0, moving_mask=shifted_line_x2 != 0)
    # shift, error, diffphase = phase_cross_correlation(line_x1, shifted_line_x2)
    if np.abs(shift[0]) > 50:
        print(f"unreasonable shift_x: {shift}, resetting to 0")
        shift[0] = 0
    if np.abs(shift[1]) > 20:
        print(f"unreasonable shift_y: {shift}, resetting to 0")
        shift[1] = 0

    initial_flow = np.full((line_x1.shape[0], line_x1.shape[1], 2), [-shift[1], -shift[0]], dtype=np.float32)
    flow = cv2.calcOpticalFlowFarneback(
        line_x1, shifted_line_x2, initial_flow,
        pyr_scale=0.8, levels=60, winsize=100,
        iterations=32, poly_n=7, poly_sigma=1.5, flags=cv2.OPTFLOW_USE_INITIAL_FLOW
    )
    u = flow[:, :, 0]
    v = flow[:, :, 1]
    avg_u = np.mean(u)
    y_shift = int(overlap_width - shift[1])
    
    shifted_x2 = shifted_x2[:, y_shift:]
    nr, nc = shifted_x2.shape
    row_coords, col_coords = np.meshgrid(np.arange(nr), np.arange(nc), indexing='ij')

    v_per_row = np.mean(v, axis=1)
    x = np.linspace(0, 1, nc-overlap_width)
    x = np.concatenate((x, np.ones(overlap_width)))
    tqdm.write(f"shift: {shift}, avg_u: {avg_u}, min: {v_per_row.min()}, max: {v_per_row.max()}, line_max: {lines_x[line_index_2].max()}")
    v_per_row = v_per_row[:, np.newaxis] * (1 - x)

    row_coords, col_coords = np.meshgrid(np.arange(shifted_x2.shape[0]), np.arange(shifted_x2.shape[1]), indexing='ij')
    warp_shifted_x2 = warp(shifted_x2, np.array([row_coords + v_per_row, col_coords]), order=0)
    print(f"warp_shifted_x2 max: {warp_shifted_x2.max()}, min: {warp_shifted_x2.min()}")
    print(f"merged_events_x max: {merged_events_x.max()}, min: {merged_events_x.min()}")


    merged_events_x = np.concatenate((merged_events_x, warp_shifted_x2), axis=1)
    # if i % 10 == 0 and i < 61:
    #     plt.imsave(os.path.join(output_folder, f"merged_events_x_{i}.png"), merged_events_x, vmin=-10, vmax=10)

np.save(os.path.join(output_folder, f"merged_shifted_final_{axis}.npy"), merged_events_x)
plt.imsave(os.path.join(output_folder, f"merged_shifted_final_{axis}.png"), merged_events_x, vmin=-10, vmax=10)
print(merged_events_x.shape)

Merging events:   0%|          | 0/9 [00:00<?, ?it/s]

shift: [2. 0.], error: 0.9999999999999989, diffphase: -1.9246578779496216e-17


Merging events:  11%|█         | 1/9 [00:00<00:06,  1.33it/s]

shift: [-1.  2.], avg_u: -0.5758962035179138, min: -6.051708221435547, max: 2.589343309402466, line_max: 32
warp_shifted_x2 max: 32, min: -37
merged_events_x max: 38, min: -31
shift: [-2.  0.], error: 0.9999999999999999, diffphase: 1.801619110624526e-17


Merging events:  22%|██▏       | 2/9 [00:01<00:05,  1.33it/s]

...

shift: [-1.  2.], avg_u: -1.4476442337036133, min: -7.77113676071167, max: 6.5061564445495605, line_max: 35
warp_shifted_x2 max: 34, min: -41
merged_events_x max: 44, min: -39
shift: [-56.  -4.], error: 0.9999999999999999, diffphase: -2.0288876973657063e-17


Merging events:  89%|████████▉ | 8/9 [00:05<00:00,  1.34it/s]

shift: [ 0. -4.], avg_u: 4.24488639831543, min: -14.19359302520752, max: 10.784485816955566, line_max: 45
warp_shifted_x2 max: 45, min: -34
merged_events_x max: 44, min: -41
shift: [-53.   0.], error: 0.9999999999999997, diffphase: -6.191529247624631e-18


Merging events: 100%|██████████| 9/9 [00:06<00:00,  1.33it/s]


shift: [-5.  4.], avg_u: -1.2528544664382935, min: -5.392996311187744, max: 9.659809112548828, line_max: 34
warp_shifted_x2 max: 33, min: -45
merged_events_x max: 45, min: -41
(2944, 2371)
